# Assessing lateral diffusion in untargeted spatial transcriptomics

VIB Spatial Catalyst, Nov 2025.

## Literature

Papers discussing diffusion in spatial transcriptomics.
- [1] [Systematic comparison of sequencing-based spatial transcriptomic methods](https://www.nature.com/articles/s41592-024-02325-3?error=cookies_not_supported&code=5cc4c3d5-ac1a-405c-b70a-8ff096e9d1a8) See section "molecule lateral diffusion".
- [2] [Nova-ST: Nano-patterned ultra-dense platform for spatial transcriptomics](https://pmc.ncbi.nlm.nih.gov/articles/PMC11384075/) See figure 3G.
- [3] [Spatiotemporal transcriptomic atlas of mouse organogenesis using DNA nanoball-patterned arrays](https://www.sciencedirect.com/science/article/pii/S0092867422003993) Stereoseq paper. Three approaches to quantize diffusion: (1) non-coding RNA (with Malat1 and Neat1), (2) mitochondriaal RNA, (3) genes specific for some cell types (Vip in Vip+ interneurons) can be used measure distance of transcripts to centroid of cells.
- [4] [Slide-seq: A Scalable Technology for Measuring Genome-Wide Expression at High Spatial Resolution](https://pmc.ncbi.nlm.nih.gov/articles/PMC6927209/)
- [5] [Highly sensitive spatial transcriptomics at near-cellular resolution with Slide-seqV2](https://pmc.ncbi.nlm.nih.gov/articles/PMC8606189/#F2) See figure 2B. Plots of the spatial expression of Hpca and Camk2a across CA1 soma and dendrites in the hippocampus.

About diffusion but not directly applicable:
- [6] [Why is no one talking about lateral diffusion in spatial sequencing?](https://www.linkedin.com/pulse/why-one-talking-lateral-diffusion-spatial-sequencing-brant-bassam-evnjf/) general comment
- [7] [From Transcripts to Cells: Dissecting Sensitivity, Signal Contamination, and Specificity in Xenium Spatial Transcriptomics](https://www.biorxiv.org/content/10.1101/2025.04.23.649965v1.full) about Xenium, inaccurate segmentation vs diffusion, SPLIT R package, using RCTD deconvolution
- [8] [Spatial genomics enables multi-modal study of clonal heterogeneity in tissues](https://pmc.ncbi.nlm.nih.gov/articles/PMC9301586/) Applicable ideas?

Miscalleanous:
- [9] [Oit3, a promising hallmark gene for targeting liver sinusoidal endothelial cells](https://www.nature.com/articles/s41392-023-01621-2) Oit3 is marker gene for LSECs? Do the LSECs form a single layer of cells? Can this be used to measure diffusion of Oit3 from the center of the nucleus of these cells, or the center of this cell line? Is it specific to LSECs, or also expressed in hepatocytes?
- [10] [xxx seq-scope xxx](https://pmc.ncbi.nlm.nih.gov/articles/PMC8238917/) See Fig 3H, genes enriched in nucleus (e.g. Malat1, Neat1 but also Mlxipl; study in liver)

This leads to the following potentially useful genes for characterizing diffusion:

- Malat1 aka Neat2, and Neat1. Long non-coding RNA, expressed only in the nucleus. See [2, 3].
- Ccn2 (brain, cortical layer 6b, see [2])
- Hpca and Camk2a (brain, see [5])
- Ptgds (brain, see [1])
- Vip (brain, in Vip+ interneurons, see [3])
- Oit3 (??, liver, see [9])

## Environment setup

Create a virtual environment with [uv](https://docs.astral.sh/uv/) in the `diffusion` subfolder of the repo:

```
uv venv --python=3.12
.venv\Scripts\activate
uv pip install s3fs pandas ipykernel
# TODO: add zarr (or spatialdata and spatialdata_plot)
# maybe: uv pip install "git+https://github.com/saeyslab/harpy.git#egg=harpy-analysis[extra]"
```

## Imports

In [1]:
import s3fs
import pandas as pd
from dataclasses import dataclass
import zarr
# import spatialdata as sd

### Create S3 filesystem

Create an S3 file system for accessing the minio server at the VIB Data Core, which houses the data for our benchmark study.

The access and secret keys for accessing the S3 server can be stored in `%USERPROFILE%\.aws\credentials`. This file looks like this:

```
    [default]
    aws_access_key_id=XXXXXXXXXXXXXXXXXXXX
    aws_secret_access_key=YYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYY
```

In [2]:
s3 = s3fs.S3FileSystem(endpoint_url="https://objectstor.vib.be")

Sanity check: list buckets and a few benchmark folders.

In [3]:
# List all buckets
print(s3.ls("/"))

# List UT benchmark folder
print(s3.ls("s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data"))

['c01-bart.lambrecht-spatial.catalyst', 'c01-charlotte.scott-spatial.catalyst', 'c01-martin.guilliams-spatial.catalyst', 'c01-yvan.saeys-spatial.catalyst', 'c03-celine.everaert-spatial.catalyst', 'c03-francis.impens-spatial.catalyst', 'c05-chris.marine-spatial.catalyst', 'c05-david.nittner-sandbox', 'ccr8e0merscope-rawdata-2025-0008', 's00-spatial.catalyst-archivebic', 's00-spatial.catalyst-evelien.van.hamme-vitessce-test', 's00-spatial.catalyst-team', 's02-bioimagingcore-team', 'spatial-hackathon-private', 'spatial-hackathon-public']
['s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed', 's00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/raw']


## Collect genes

Iterate over the processed data of all experiments in the benchmark, and collect information on genes that could be useful for characterizing diffusion.

In [4]:
@dataclass
class Experiment:
    technology: str
    species: str
    tissue: str
    gene_counts_path: str  # S3 path to CSV file with the gene counts for this experiment

In [5]:
S3_PATH_PREFIX = "s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/"

experiments = [Experiment("BMKMANU S1000", "mouse", "brain WT", S3_PATH_PREFIX + "BMK_S1000/exp74/harpy/gene_total_counts.csv"),
               
               Experiment("BMKMANU S3000", "mouse", "brain WT", S3_PATH_PREFIX + "BMK_S3000/Exp99/harpy/gene_total_counts.csv"),
               Experiment("BMKMANU S3000", "mouse", "liver WT", S3_PATH_PREFIX + "BMK_S3000/Exp100/harpy/gene_total_counts.csv"),
               Experiment("BMKMANU S3000", "mouse", "lung WT", S3_PATH_PREFIX + "BMK_S3000/Exp101/harpy/gene_total_counts.csv"),
               Experiment("BMKMANU S3000", "human", "lung tumor", S3_PATH_PREFIX + "BMK_S3000/Exp102/harpy/gene_total_counts.csv"),

               Experiment("Curio Seeker", "mouse", "brain WT", S3_PATH_PREFIX + "Curio_seeker/SC11/harpy/gene_total_counts.csv"),
               Experiment("Curio Seeker", "mouse", "liver WT", S3_PATH_PREFIX + "Curio_seeker/SC13/harpy/gene_total_counts.csv"),
               Experiment("Curio Seeker", "mouse", "lung WT", S3_PATH_PREFIX + "Curio_seeker/SC12/harpy/gene_total_counts.csv"),
               Experiment("Curio Seeker", "human", "lung tumor", S3_PATH_PREFIX + "Curio_seeker/SC14/harpy/gene_total_counts.csv"),

               Experiment("Visium HD FFPE", "mouse", "brain WT", S3_PATH_PREFIX + "Visium_HD/SCA003/harpy/gene_total_counts.csv"),
               Experiment("Visium HD FFPE", "mouse", "liver WT", S3_PATH_PREFIX + "Visium_HD/SCA002/harpy/gene_total_counts.csv"),
               Experiment("Visium HD FFPE", "mouse", "lung WT", S3_PATH_PREFIX + "Visium_HD/SCA001/harpy/gene_total_counts.csv"),
               Experiment("Visium HD FFPE", "mouse", "colon WT", S3_PATH_PREFIX + "Visium_HD/SCA005/harpy/gene_total_counts.csv"),
               Experiment("Visium HD FFPE", "human", "lung tumor", S3_PATH_PREFIX + "Visium_HD/SCA010/harpy/gene_total_counts.csv"),

               Experiment("Visium HD FF", "mouse", "brain WT", S3_PATH_PREFIX + "Visium_HD/SCA019/harpy/gene_total_counts.csv"),
               Experiment("Visium HD FF", "mouse", "liver WT", S3_PATH_PREFIX + "Visium_HD/SCA018/harpy/gene_total_counts.csv"),
               Experiment("Visium HD FF", "mouse", "lung WT", S3_PATH_PREFIX + "Visium_HD/SCA017/harpy/gene_total_counts.csv"),
               Experiment("Visium HD FF", "human", "lung tumor", S3_PATH_PREFIX + "Visium_HD/SCA020/harpy/gene_total_counts.csv"),

               Experiment("Visium FFPE", "mouse", "brain WT", S3_PATH_PREFIX + "Visium/SCA016/harpy/gene_total_counts.csv"),
               Experiment("Visium FFPE", "mouse", "liver WT", S3_PATH_PREFIX + "Visium/SCA014/harpy/gene_total_counts.csv"),
               Experiment("Visium FFPE", "mouse", "lung WT", S3_PATH_PREFIX + "Visium/SCA013/harpy/gene_total_counts.csv"),
               Experiment("Visium FFPE", "human", "lung tumor", S3_PATH_PREFIX + "Visium/SCA015/harpy/gene_total_counts.csv"),

               # TODO: Curio Trekker
               # TODO: Stereoseq
               # TODO: Nova-ST
               # TODO: Visium HD 3'
               ]

In [6]:
GENERAL_GENES_OF_INTEREST = ["MALAT1", "NEAT1"]
BRAIN_GENES_OF_INTEREST = ["CCN2", "PTGDS", "VIP", "HPCA", "CAMK2A", "SLC17A7"]
LIVER_GENES_OF_INTEREST = ["OIT3"]  # Idea Frank, gene for targeting LSECs (Liver Sinusoidal Endothelial Cells) ?? 

In [7]:
# Note: beware of different gene name capitalization conventions (human vs non-human). We convert to upper case during comparisons.
def collect_genes(df: pd.DataFrame,
                  brain_genes_of_interest: list[str]) -> dict[str, float]: 
    genes = set(gene.upper() for gene in df["gene"])
    mitochondrial_genes = [gene for gene in genes if gene.startswith("MT-")]
    genes_of_interest = brain_genes_of_interest + mitochondrial_genes
    # TODO: assert that all genes of interest are all caps at this point
    detected_genes = {}
    for gene in genes_of_interest:
        found = gene in genes
        if found:
            counts_list = list(df[df.gene.apply(lambda x: x.upper()) == gene].total_counts)
            assert len(counts_list) == 1
            counts = counts_list[0]
            # print(f"{gene} found with {counts} counts.")
            detected_genes[gene] = counts
        # else:
        #     print(f"{gene} not found.")
    return detected_genes

In [8]:
def build_diffusion_related_genes_overview(experiments: list[Experiment],
                                           genes_of_interest: list[str]) -> list[tuple[Experiment, dict[str, float]]]:
    overview = []
    for experiment in experiments:
        s3_path = experiment.gene_counts_path
        print(f"Reading {s3_path}")
        with s3.open(s3_path, "rb") as f:
            df = pd.read_csv(f)
        detected_genes = collect_genes(df, genes_of_interest)
        overview.append((experiment, detected_genes))
    return overview

# TODO: what is total_counts_corrected?

In [9]:
def build_diffusion_related_genes_dataframe(experiments: list[Experiment],
                                            genes_of_interest: list[str]) -> pd.DataFrame:
    overview = build_diffusion_related_genes_overview(experiments, genes_of_interest)
    all_detected_genes = sorted(list({gene for overview_entry in overview for gene in overview_entry[1].keys() }))

    ds = []
    for overview_entry in overview:
        e, g = overview_entry
        d = {}
        d["technology"] = e.technology
        d["species"] = e.species
        d["tissue"] = e.tissue
        d["path"] = e.gene_counts_path
        for gene in all_detected_genes:
            d[gene] = g.get(gene, 0.0)
        ds.append(d)

    return pd.DataFrame.from_dict(ds)

In [10]:
df = build_diffusion_related_genes_dataframe(experiments, GENERAL_GENES_OF_INTEREST + BRAIN_GENES_OF_INTEREST + LIVER_GENES_OF_INTEREST)
df

Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp102/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC11/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC13/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC12/harpy/gene_total_counts.csv
Reading s00-spatial.catalyst-team/2024-07-UTBenchmark-S

,technology,species,tissue,path,CAMK2A,CCN2,HPCA,MALAT1,MT-ATP6,MT-ATP8,...,MT-ND3,MT-ND4,MT-ND4L,MT-ND5,MT-ND6,NEAT1,OIT3,PTGDS,SLC17A7,VIP
0,BMKMANU S1000,mouse,brain WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,19381.0,1412.0,23593.0,8740.0,933529.0,21689.0,...,287834.0,494326.0,16114.0,66520.0,166250.0,4461.0,5.0,43122.0,22154.0,3273.0
1,BMKMANU S3000,mouse,brain WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,21056.0,1620.0,60065.0,102888.0,2717044.0,132538.0,...,1073474.0,1387587.0,64958.0,248324.0,442881.0,2863.0,13.0,174820.0,19897.0,4173.0
2,BMKMANU S3000,mouse,liver WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,1016.0,528.0,44.0,24643.0,4649872.0,49535.0,...,1142552.0,2070725.0,36378.0,197477.0,579877.0,16688.0,6034.0,971.0,4.0,2.0
3,BMKMANU S3000,mouse,lung WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,382.0,9844.0,78.0,31354.0,567156.0,7152.0,...,184931.0,367802.0,6458.0,32592.0,103192.0,25269.0,89.0,182.0,156.0,20.0
4,BMKMANU S3000,human,lung tumor,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,274.0,68544.0,655.0,89236.0,229668.0,3524.0,...,180240.0,282847.0,9140.0,16979.0,21020.0,47443.0,213.0,2008.0,46.0,8.0
5,Curio Seeker,mouse,brain WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,1698.0,126.0,2926.0,11328.0,17.0,1.0,...,18.0,17247.0,1.0,1859.0,2057.0,1038.0,0.0,2919.0,752.0,37.0
6,Curio Seeker,mouse,liver WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,2.0,17.0,3.0,1101.0,11.0,0.0,...,17.0,40410.0,1.0,2368.0,2626.0,1932.0,75.0,75.0,2.0,0.0
7,Curio Seeker,mouse,lung WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,7.0,553.0,6.0,4218.0,5.0,0.0,...,5.0,4607.0,0.0,523.0,655.0,2527.0,3.0,2.0,1.0,0.0
8,Curio Seeker,human,lung tumor,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,6.0,376.0,0.0,4877.0,880.0,11.0,...,241.0,349.0,1.0,47.0,84.0,1868.0,3.0,6.0,5.0,1.0
9,Visium HD FFPE,mouse,brain WT,s00-spatial.catalyst-team/2024-07-UTBenchmark-...,453706.0,4946.0,129712.0,0.0,721261.0,510135.0,...,66835.0,275174.0,208525.0,271547.0,39815.0,0.0,23.0,63417.0,137411.0,7307.0


In [11]:
# Accessing Spatiadatal via S3?
# uv pip install spatialdata spatialdata_plot

# TODO: access the AnnData object (we want the locations of all transcripts for a given gene)
# TODO: access the segmentation mask image (we want the masks for the nuclei)


In [12]:
import zarr
root = zarr.open("s3://s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr",
                 storage_options = {'client_kwargs': {'endpoint_url': "https://objectstor.vib.be"}})
print(list(root.groups()))
print(root["images"])
print(root["labels"])
print(root["shapes"])
print(root["tables"])


[('', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>>), ('shapes', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/shapes>), ('images', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/images>), ('tables', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/tables>), ('labels', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/labels>)]
<Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/images>
<Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/proces

c:\git\2024-07-UTBenchmark-SpC\diffusion\.venv\Lib\site-packages\zarr\core\group.py:3530: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [13]:
print(list(root['labels'].groups()))

[('exp99_level_18_labels', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/labels/exp99_level_18_labels>), ('labels', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/labels>), ('exp99_level_1_labels', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/labels/exp99_level_1_labels>), ('exp99_level_2_labels', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/labels/exp99_level_2_labels>), ('exp99_level_4_labels', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/labels/exp99_level_4_labels>), ('exp99_level_5_labels', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-te

In [24]:
print(root.tree(level=2))

c:\git\2024-07-UTBenchmark-SpC\diffusion\.venv\Lib\site-packages\zarr\core\group.py:3530: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(
c:\git\2024-07-UTBenchmark-SpC\diffusion\.venv\Lib\site-packages\zarr\core\group.py:3530: ZarrUserWarning: Object at shapes.parquet is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/
└── 
    └── 
        └── 
            ├── images
            ├── images
            ├── images
            │   ├── exp99_he
            │   ├── exp99_he
            │   │   ├── 0 (3, 20000, 19954) uint8
            │   │   ├── 1 (3, 10000, 9977) uint8
            │   │   ├── 2 (3, 5000, 4988) uint8
            │   │   ├── 3 (3, 2500, 2494) uint8
            │   │   ├── 4 (3, 1250, 1247) uint8
            │   │   └── exp99_he
            │   ├── images
            │   └── images
            │       ├── exp99_he
            │       └── images
            ├── labels
            ├── labels
            ├── labels
            │   ├── exp99_level_18_labels
            │   ├── exp99_level_18_labels
            │   │   ├── 0 (20296, 20273) uint16
            │   │   └── exp99_level_18_labels
            │   ├── exp99_level_1_labels
            │   ├── exp99_level_1_labels
            │   │   ├── 0 (19996, 19949) uint32
            │   │   └── exp99_level_1_labels
            │   ├── exp99_level_2_labels
            │   ├── exp99_level_2_labels
            │   │   ├── 0 (20013, 19967) uint32
            │   │   └── exp99_level_2_labels
            │   ├── exp99_level_3_labels
            │   ├── exp99_level_3_labels
            │   │   ├── 0 (20031, 19986) uint32
            │   │   └── exp99_level_3_labels
            │   ├── exp99_level_4_labels
            │   ├── exp99_level_4_labels
            │   │   ├── 0 (20048, 20005) uint32
            │   │   └── exp99_level_4_labels
            │   ├── exp99_level_5_labels
            │   ├── exp99_level_5_labels
            │   │   ├── 0 (20066, 20025) uint32
            │   │   └── exp99_level_5_labels
            │   ├── exp99_level_6_labels
            │   ├── exp99_level_6_labels
            │   │   ├── 0 (20084, 20044) uint16
            │   │   └── exp99_level_6_labels
            │   ├── exp99_level_7_labels
            │   ├── exp99_level_7_labels
            │   │   ├── 0 (20102, 20063) uint16
            │   │   └── exp99_level_7_labels
            │   ├── exp99_level_9_labels
            │   ├── exp99_level_9_labels
            │   │   ├── 0 (20137, 20101) uint16
            │   │   └── exp99_level_9_labels
            │   ├── exp99_tissue_mask
            │   ├── exp99_tissue_mask
            │   │   ├── 0 (17401, 16293) uint8
            │   │   └── exp99_tissue_mask
            │   ├── labels
            │   └── labels
            │       ├── exp99_level_18_labels
            │       ├── exp99_level_1_labels
            │       ├── exp99_level_2_labels
            │       ├── exp99_level_3_labels
            │       ├── exp99_level_4_labels
            │       ├── exp99_level_5_labels
            │       ├── exp99_level_6_labels
            │       ├── exp99_level_7_labels
            │       ├── exp99_level_9_labels
            │       ├── exp99_tissue_mask
            │       └── labels
            ├── shapes
            ├── shapes
            ├── shapes
            │   ├── exp99_capture_area
            │   ├── exp99_capture_area
            │   │   └── exp99_capture_area
            │   ├── exp99_level_1
            │   ├── exp99_level_1
            │   │   └── exp99_level_1
            │   ├── exp99_level_18
            │   ├── exp99_level_18
            │   │   └── exp99_level_18
            │   ├── exp99_level_2
            │   ├── exp99_level_2
            │   │   └── exp99_level_2
            │   ├── exp99_level_3
            │   ├── exp99_level_3
            │   │   └── exp99_level_3
            │   ├── exp99_level_4
            │   ├── exp99_level_4
            │   │   └── exp99_level_4
            │   ├── exp99_level_5
            │   ├── exp99_level_5
            │   │   └── exp99_level_5
            │   ├── exp99_level_6
            │   ├── exp99_level_6
            │   │   └── exp99_level_6
            │   ├── exp99_level_7
            │   ├── exp99_level_7
            │   │   └── exp99_level_7
            │   ├── exp99_level_9
            │   ├── exp99_leve

In [21]:
print(list(root['tables'].groups()))

[('tables', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/tables>), ('level_2', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/tables/level_2>), ('level_18', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/tables/level_18>), ('level_1', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/tables/level_1>), ('level_3', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/tables/level_3>), ('level_4', <Group <FsspecStore(S3FileSystem, s00-spatial.catalyst-team/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/harpy/sdata.zarr)>/tables/level_4>), ('level_5', <Group <Fs

From the stereoseq paper:

_(1) known nuclear localized long non-coding RNAs Malat1 and Neat1; (2) Mitochondrial RNAs and (3) Vip transcripts that specifically expressed in Vip+ interneurons. For (1) and (2), we sampled 100 cells from all the segmented cells, and calculated the mean distance between each transcript to the centroid of each cell. After repeating 50 times, the distribution of (1) and (2) were plot. For (3), we compared the lateral diffusion of Stereo-seq with published smFISH data downloaded from Linnarsson Lab website (http://linnarssonlab.org/osmFISH/). Vip+ interneurons were selected either based on cell segmentation-based clustering (Stereo-seq) or the reported annotations (smFISH). The diffusion distance was further determined based on the distance between Vip transcripts with the centroid of each Vip+ interneurons._

Let's try (1) and (2). Start with approach (1)

Steps:
1. read H&E image
2. segment nuclei from H&E    <--- PROBLEM, e.g. the BMK S3000 H&E has circular dots from the chip and looks very ugly, not sure that we can segment it actually
3. read Malat1 and Neat1 transcript positions from table
4. read mitochondrial gene positions from table
5. assign transcripts to closest nucleus
6. sample distance distribution of nuclear and of mitochondrial RNA? Sample as in the stereoseq paper? (but this does not really yield a diffusion distance _value_.) Alternatively: perhaps draw boxplot of distance of Malat1 and Neat1 transcripts that are _outside_ the nucleus, it average is a lower bound of the diffusion?
Note: exclude transcripts that are outside the tissue to avoid most outliers?

In [1]:
import spatialdata as sd
sdata = sd.read_zarr(r"C:\git\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\Exp99\harpy\sdata.zarr")
sdata

c:\git\2024-07-UTBenchmark-SpC\diffusion\.venv\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\git\2024-07-UTBenchmark-SpC\diffusion\.venv\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
C:\Users\FRVER\AppData\Local\Temp\ipykernel_25152\3053745480.py:2: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.

SpatialData object, with associated Zarr store: C:\git\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\Exp99\harpy\sdata.zarr
├── Images
│     └── 'exp99_he': DataTree[cyx] (3, 20000, 19954), (3, 10000, 9977), (3, 5000, 4988), (3, 2500, 2494), (3, 1250, 1247)
├── Labels
│     ├── 'exp99_level_1_labels': DataArray[yx] (19996, 19949)
│     ├── 'exp99_level_2_labels': DataArray[yx] (20013, 19967)
│     ├── 'exp99_level_3_labels': DataArray[yx] (20031, 19986)
│     ├── 'exp99_level_4_labels': DataArray[yx] (20048, 20005)
│     ├── 'exp99_level_5_labels': DataArray[yx] (20066, 20025)
│     ├── 'exp99_level_6_labels': DataArray[yx] (20084, 20044)
│     ├── 'exp99_level_7_labels': DataArray[yx] (20102, 20063)
│     ├── 'exp99_level_9_labels': DataArray[yx] (20137, 20101)
│     ├── 'exp99_level_18_labels': DataArray[yx] (20296, 20273)
│     └── 'exp99_tissue_mask': DataArray[yx] (17401, 16293)
├── Shapes
│     ├── 'exp99_capture_area': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'exp99_l

In [ ]:
from napari_spatialdata import Interactive
Interactive(sdata)

2025-11-20 13:14:51.149 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.


2025-11-20 13:15:31.519 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:15:31.520 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:15:41.257 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:15:41.262 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:15:46.949 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:15:46.955 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:16:20.451 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:16:20.451 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:16:30.323 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-20 13:16:30.333 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updati